<a href="https://colab.research.google.com/github/terry0809000/Artificial-Intelligence-KCL/blob/main/7PAVAIHA_2026_Practical_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practical 3: Particle Swarm Optimisation for Feature Selection

## IMPORTANT

Please save your own copy of this workbook and work off that. Click on `File` and select `Save a copy in Drive`

Then, click on `Runtime` above and if highlighted, click on `Restart session`

Finally, click on `edit` above and click on `Clear all outputs`

#### Compute Power

This practical will only require CPU runtime.

Step 1: Open Runtime Settings
*   Click Runtime in the top menu.
*   Click Change runtime type.

Step 2: Select CPU, then click on Save

<hr style="border:1px solid black"> </hr>

## Introduction

In today's lab, we will learn how to perform feature selection using **Particle Swarm Optimisation (PSO)**, utilising the [pyswarms](https://pyswarms.readthedocs.io/en/latest/index.html) toolkit.

**Question:** Why might we want to perform feature selection?
- What are some advantages and disadvantages of using it in machine learning pipelines?

## Step 1: Install and importing relevant modules

In [ ]:
pip install pyswarms

In [ ]:
# Import modules
import numpy as np
import time

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Import PySwarms
import pyswarms as ps

## Step 2: Create dataset with redundant features

Use `sklearn`'s __[make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html)__ function to create a dataset with a large number of redundant features.

**Task:** Add comments to the code explaining the core steps in detail.
- Focus on what each parameter and step is doing, not just what the code says.
- You can use an AI assistant to help support your understanding.

In [ ]:
# Define the number of features for each type
n_features = 200
n_informative = 20
n_redundant = 100
n_repeated = 50
n_useless = 30

# Create Labels
informative_labels = [f'Informative {ii}' for ii in range(1, n_informative + 1)]
redundant_labels = [f'Redundant {ii}' for ii in range(n_informative + 1, n_informative + n_redundant + 1)]
repeated_labels = [f'Repeated {ii}' for ii in range(n_informative + n_redundant+ 1, n_informative + n_redundant + n_repeated + 1)]
useless_labels = [f'Useless {ii}' for ii in range(n_informative + n_redundant + n_repeated + 1, n_features + 1)]
labels = informative_labels + redundant_labels + repeated_labels + useless_labels

# Get data
X, y = make_classification(n_samples = 5000, n_features = n_features,
                           n_informative = n_informative,
                           n_redundant = n_redundant , n_repeated = n_repeated,
                           n_clusters_per_class = 2, class_sep = 0.5, flip_y = 0.05,
                           random_state = 42, shuffle = False)


We now split the data into training and test sets.

**Question** Why do we need to do this? To obtain an unbiased estimate of model performance on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

The role of the code in the next cell is to standardise the data.

**Question** What does standardisation do to the data, and why might this be important for feature selection? Standardisation rescales features to have mean zero and unit variance, ensuring that all variables are on a comparable scale. This is particularly important for feature selection methods such as LASSO, where penalties depend on coefficient magnitude; without standardisation, variables with larger scales may be unfairly favoured, leading to biased or unstable feature selection.

In [ ]:
from sklearn import preprocessing

scaler = preprocessing.StandardScaler().fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

## Step 3: Set-up the objective function

We’ll be using the optimizer `pyswarms.discrete.BinaryPSO` to perform feature subset selection.

For a Binary PSO, the position of the particles are expressed in two terms: 1 or 0 (or on and off).

Mathematically, this is defined as:

$x=[x_{1},x_{2},x_{3},…,x_{d}]$

where:

$x_{i}\in {0,1}$

The objective function we will be using is taken from this __[paper](https://www.sciencedirect.com/science/article/abs/pii/S1568494613001361)__, and is define as:

$f(X)=\alpha(1−P)+(1−\alpha)(1− \dfrac{N_{f}}{N_{t}})$

Where
- $\alpha$ is a hyperparameter tradeoffs the performance of classifier
- $P$, with the ratio of the size of the feature subset
- $N_{f}$ with respect to the total number of features $N_{t}$

<hr style="border:1px solid black"> </hr>

#### Note

The twin goals of feature selection are to maximise model performance while minimising the number of selected features.

The term $\alpha(1 - P)$ represents the classification error (i.e. how well the model performs), while $(1 - \alpha)\left(1 - \dfrac{N_f}{N_t}\right)$ represents the proportion of selected features relative to the total number available.

By adjusting $\alpha$, we are effectively trading off between classification performance and selecting fewer features.

However, we need to be careful. If $\alpha$ is weighted too heavily towards performance, we may end up with more features than training instances.

**Question**
What is this phenomenon called, and what effect might it have on classifier performance?

<hr style="border:1px solid black"> </hr>

We now need need to define a function which calcualtes the objective function per particle.

Our function `f_per_particle` a binary mask and $\alpha$ and returns the object loss score for a *single* particle

In [ ]:
# Create an instance of the classifier
classifier = RandomForestClassifier(max_depth=2,n_estimators=10)

# Define objective function
def f_per_particle(m, alpha):

    X_train_subset, X_test_subset, y_train_subset, y_test_subset = train_test_split(X_train, y_train, test_size=0.33)

    # Get the subset of the features from the binary mask
    if np.count_nonzero(m) == 0:
        X_train_masked = X_train_subset
        X_test_masked = X_test_subset
    else:
        X_train_masked = X_train_subset[:,m==1]
        X_test_masked = X_test_subset[:,m==1]

    # Perform classification and store performance in P
    classifier.fit(X_train_masked, y_train_subset)
    P = (classifier.predict(X_test_masked) == y_test_subset).mean()

    # Compute for the objective function
    j = (alpha * (1.0 - P)
        + (1.0 - alpha) * (1 - (X_train_masked.shape[1] /X_train_subset.shape[1])))

    return j

Next we define the higher level objective function to be evaluated in pyswarms built in optimiser.

`f` returns the object loss score for *all* particles

In [ ]:
def f(M, alpha=0.5):
    """Higher-level method to do classification in the
    whole swarm.

    Inputs
    ------
    M: numpy.ndarray of shape (n_particles, dimensions)
        The swarm that will perform the search

    Returns
    -------
    numpy.ndarray of shape (n_particles, )
        The computed loss for each particle
    """

    n_particles = M.shape[0]

    j = [f_per_particle(M[i], alpha) for i in range(n_particles)]

    return np.array(j)

## Step 4: Set-up the optimisation parameters and run the selections algorithm

In the `options` dictionary below:
- `k`represents the neighbors to be considered when calculating the best known position of the swarm
- `p`represents a distance metric used in the optimisation algorithm.

**Question** What are the purpose of the other parameters?

### Task: Track Runtime

Update the code below so that you can measure how long the optimiser takes to run and report this result.

In [ ]:
# Initialize swarm, arbitrary
options = {'c1': 1.8, 'c2': 0.5, 'w':0.5, 'k': 20, 'p':2}

# Call instance of PSO
dimensions = X.shape[1] # dimensions should be the number of features

optimizer = ps.discrete.BinaryPSO(n_particles=25, dimensions=dimensions, options=options)

# Perform optimization
cost, pos = optimizer.optimize(f, iters=10, verbose=2)
selected = np.count_nonzero(pos)
print("Optimiser retained " + str(selected)   + " features")

## Step 5: Evalute the performamce of the selected features

We evaluate the effectiveness of the selected features, we can use two Random Forest Classifiers, one trained on the full feature set, the other trained on the selected feature sets, and compare the outputs

In [ ]:
c1 = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)
c1.fit(X_train, y_train)

full_performance = (c1.predict(X_test) == y_test).mean()
print('Full Feature set performance: %.3f' % (full_performance))

c2 = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)
c2.fit(X_train[:,pos==1], y_train)

subset_performance = (c2.predict(X_test[:,pos==1]) == y_test).mean()
print('Subset Feature set performance: %.3f' % (subset_performance))

## Exercise 1

### Exploration Task

Investigate how changing key parameters affects both the performance and running time of the optimiser.

Parameters to experiment with:
- `alpha`
- Swarm options
- Number of particles
- Number of optimiser iterations

**Systematic approach:**
- Change **one parameter at a time** while keeping others constant  
- Record the effect on performance and runtime  
- Compare your results across different settings  

---

**Parameter Ranges**

To keep runtime reasonable, use the following ranges:

- `n_particles`: **10, 20, 50**
- `iters`: **10, 20, 30**
- `alpha`: **0.3, 0.6, 0.9**
- `c1`, `c2`, `w`: *small adjustments only* (±0.25 from original values)

---

### Reflection:
Did you observe any trade-off between performance and computational cost?

In [13]:
import numpy as np
import time
import pandas as pd

# Base parameters (from previous cells or problem description)
base_n_particles = 25
base_iters = 10
base_alpha = 0.5 # Default in f() function
base_options = {'c1': 1.8, 'c2': 0.5, 'w': 0.5, 'k': 20, 'p': 2}
dimensions = X_train.shape[1] # Use X_train.shape[1] to ensure consistency with the current dataset

results = []

# --- Helper function to run PSO and evaluate performance ---
def run_pso_experiment(n_particles, iters, alpha, options_dict, experiment_name):
    # Print experiment details for tracking
    print(f"Running experiment: {experiment_name}, Params: n_p={n_particles}, iters={iters}, alpha={alpha}, c1={options_dict['c1']:.2f}, c2={options_dict['c2']:.2f}, w={options_dict['w']:.2f}")
    start_time = time.time()

    current_options = options_dict.copy()
    # Adjust k if n_particles is too small for the original k
    if n_particles <= current_options['k']:
        current_options['k'] = max(1, n_particles - 1) # Ensure k is at least 1, and less than n_particles
        print(f"  Adjusted 'k' to {current_options['k']} for n_particles={n_particles} to prevent IndexError.")

    # Initialize PSO optimizer with current parameters
    optimizer = ps.discrete.BinaryPSO(n_particles=n_particles, dimensions=dimensions, options=current_options)

    # Perform optimization; use a lambda to pass alpha to the objective function `f`
    cost, pos = optimizer.optimize(lambda m: f(m, alpha), iters=iters, verbose=0) # verbose=0 to reduce output

    runtime = time.time() - start_time
    selected_features_count = np.count_nonzero(pos)

    # Evaluate performance of selected features using a new RandomForestClassifier instance for this evaluation
    # This classifier is for final evaluation of the selected features, not the one used within PSO's objective.
    eval_classifier = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)

    subset_performance_exp = 0.0 # Default if no features or other issues

    if selected_features_count == 0:
        # If no features were selected by PSO, assign a 'random guess' accuracy for binary classification.
        # This prevents errors from training on an empty feature set.
        subset_performance_exp = 0.5
        print("  WARNING: PSO selected 0 features. Performance set to 0.5 (random guess).")
    else:
        # Train and evaluate the classifier on the selected subset of features
        eval_classifier.fit(X_train[:,pos==1], y_train)
        subset_performance_exp = (eval_classifier.predict(X_test[:,pos==1]) == y_test).mean()

    # Store results
    results.append({
        'Experiment': experiment_name,
        'n_particles': n_particles,
        'iters': iters,
        'alpha': alpha,
        'c1': options_dict['c1'],
        'c2': options_dict['c2'],
        'w': options_dict['w'],
        'Runtime (s)': runtime,
        'Selected Features': selected_features_count,
        'Subset Performance': subset_performance_exp,
        'Optimizer Cost': cost
    })
    print(f"  Runtime: {runtime:.2f}s, Sel.Feat: {selected_features_count}, Perf: {subset_performance_exp:.3f}, Cost: {cost:.3f}")
    print("-" * 50)


# --- Run experiments ---

print("--- Running Experiments for PSO Parameter Tuning ---")
print(f"Full Feature set performance (for reference): {full_performance:.3f}")
print("-" * 50)

# Baseline Run
print("--- Baseline Run (re-running with defined base parameters) ---")
run_pso_experiment(base_n_particles, base_iters, base_alpha, base_options, 'Baseline')

# Vary n_particles
print("\n--- Varying n_particles ---")
for n_p in [10, 20, 50]:
    if n_p == base_n_particles:
        continue # Already covered by baseline
    run_pso_experiment(n_p, base_iters, base_alpha, base_options, f'Vary n_particles (n_p={n_p})')

# Vary iters
print("\n--- Varying iters ---")
for i in [10, 20, 30]:
    if i == base_iters:
        continue # Already covered by baseline
    run_pso_experiment(base_n_particles, i, base_alpha, base_options, f'Vary iters (iters={i})')

# Vary alpha
print("\n--- Varying alpha ---")
# The problem statement for alpha values [0.3, 0.6, 0.9] does not include 0.5.
# So all these will be new runs.
for a in [0.3, 0.6, 0.9]:
    if a == base_alpha: # Skip if base_alpha happens to be in the list, though unlikely here
        continue
    run_pso_experiment(base_n_particles, base_iters, a, base_options, f'Vary alpha (alpha={a})')


# Vary c1
print("\n--- Varying c1 (Cognitive coefficient) ---")
# Original c1: 1.8
for c_val in [1.55, 1.8, 2.05]: # Values: base-0.25, base, base+0.25
    if c_val == base_options['c1']:
        continue # Already covered by baseline
    new_options = base_options.copy()
    new_options['c1'] = c_val
    run_pso_experiment(base_n_particles, base_iters, base_alpha, new_options, f'Vary c1 (c1={c_val:.2f})')

# Vary c2
print("\n--- Varying c2 (Social coefficient) ---")
# Original c2: 0.5
for c_val in [0.25, 0.5, 0.75]: # Values: base-0.25, base, base+0.25
    if c_val == base_options['c2']:
        continue # Already covered by baseline
    new_options = base_options.copy()
    new_options['c2'] = c_val
    run_pso_experiment(base_n_particles, base_iters, base_alpha, new_options, f'Vary c2 (c2={c_val:.2f})')

# Vary w
print("\n--- Varying w (Inertia weight) ---")
# Original w: 0.5
for w_val in [0.25, 0.5, 0.75]: # Values: base-0.25, base, base+0.25
    if w_val == base_options['w']:
        continue # Already covered by baseline
    new_options = base_options.copy()
    new_options['w'] = w_val
    run_pso_experiment(base_n_particles, base_iters, base_alpha, new_options, f'Vary w (w={w_val:.2f})')


# --- Display results ---
print("\n" * 2)
print("--- Experiment Results Summary ---")
results_df = pd.DataFrame(results)
# Reorder columns for better readability if needed
ordered_cols = ['Experiment', 'n_particles', 'iters', 'alpha', 'c1', 'c2', 'w',
                'Runtime (s)', 'Selected Features', 'Subset Performance', 'Optimizer Cost']
results_df = results_df[ordered_cols]
print(results_df.round(3).to_markdown(index=False))


--- Running Experiments for PSO Parameter Tuning ---
Full Feature set performance (for reference): 0.650
--------------------------------------------------
--- Baseline Run (re-running with defined base parameters) ---
Running experiment: Baseline, Params: n_p=25, iters=10, alpha=0.5, c1=1.80, c2=0.50, w=0.50
  Runtime: 19.49s, Sel.Feat: 121, Perf: 0.693, Cost: 0.335
--------------------------------------------------

--- Varying n_particles ---
Running experiment: Vary n_particles (n_p=10), Params: n_p=10, iters=10, alpha=0.5, c1=1.80, c2=0.50, w=0.50
  Adjusted 'k' to 9 for n_particles=10 to prevent IndexError.
  Runtime: 8.44s, Sel.Feat: 130, Perf: 0.688, Cost: 0.324
--------------------------------------------------
Running experiment: Vary n_particles (n_p=20), Params: n_p=20, iters=10, alpha=0.5, c1=1.80, c2=0.50, w=0.50
  Adjusted 'k' to 19 for n_particles=20 to prevent IndexError.
  Runtime: 15.93s, Sel.Feat: 115, Perf: 0.684, Cost: 0.361
---------------------------------------

## Exercise #2

### Feature Selection on Breast Cancer Dataset

In this exercise, you will use the *Breast Cancer Wisconsin Diagnostic Dataset*.

- Features are computed from digitised images of fine needle aspirates (FNA) of breast masses  
- Each sample is labelled as:
  - **Malignant (0)**
  - **Benign (1)**  
- The dataset can be imported directly from `sklearn`

---

#### Part 1 — Simple Filter Method

Apply a basic **filter-based feature selection method** and evaluate its impact.

**Tasks:**
1. Implement and apply a filter method

2. Train a **Random Forest Classifier** on:
   - The **full feature set**
   - The **reduced feature set**

3. Compare:
   - Model accuracy
   - Number of selected features

---

#### Part 2 — Particle Swarm Optimisation (PSO)

Re-implement **Particle Swarm Optimisation (PSO)** for feature selection using the full dataset.

**Tasks:**
1. Apply PSO to select an optimal subset of features  
2. Use a **systematic approach** to tune hyperparameters (e.g. grid search or controlled experimentation):
   - Number of particles  
   - Number of iterations  
   - Cognitive and social coefficients  

3. Train the same Random Forest model using:
   - Features selected by PSO  

4. Record:
   - Model performance (e.g. accuracy)
   - Number of selected features  

---

#### Comparison Table

Fill in the table below:

| Method             | Accuracy| # Features| Notes|
|--------------------|---------|-----------|------|
| Full Dataset       |         | 30        |      |
| Filter Method      |         |           |      |
| PSO Selection      |         |           |      |

---

#### Reflection

Consider the following questions:

**Does feature selection improve performance on this dataset?**  
   - Did accuracy increase, decrease, or stay similar?

**Did one method performed better (Filter vs PSO)?**  
   - Provide evidence from your results  
   - Why do you think this was the case?  

**Trade-offs:**  
   - Which method is more computationally expensive?  
   - Which method is easier to interpret?

### Part 1 — Simple Filter Method

In [14]:
# Load the dataset (already done in cell GQMYx24IiYiH)
# X, y = data.data, data.target

# Split data into training and testing sets
from sklearn.model_selection import train_test_split
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize the data
from sklearn import preprocessing
scaler_bc = preprocessing.StandardScaler().fit(X_train_bc)
X_train_bc_scaled = scaler_bc.transform(X_train_bc)
X_test_bc_scaled = scaler_bc.transform(X_test_bc)

print(f"Original number of features: {X_train_bc_scaled.shape[1]}")

Original number of features: 30


In [18]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier

# --- Filter Method: Select K Best Features ---
# Using f_classif (ANOVA F-value) to select features, aiming for a reasonable number like half the original
k_features_filter = int(X_train_bc_scaled.shape[1] / 2) # e.g., 15 features for breast cancer dataset
selector = SelectKBest(f_classif, k=k_features_filter)

X_train_filtered = selector.fit_transform(X_train_bc_scaled, y_train_bc)
X_test_filtered = selector.transform(X_test_bc_scaled)

selected_features_mask_filter = selector.get_support()
num_selected_features_filter = np.sum(selected_features_mask_filter)

print(f"Number of features selected by Filter Method: {num_selected_features_filter}")

# --- Train and evaluate Random Forest Classifier on Full Feature Set ---
rf_full = RandomForestClassifier(random_state=42)
rf_full.fit(X_train_bc_scaled, y_train_bc)
full_accuracy_bc = rf_full.score(X_test_bc_scaled, y_test_bc)

print(f"Accuracy with Full Feature Set: {full_accuracy_bc:.3f}")

# --- Train and evaluate Random Forest Classifier on Filtered Feature Set ---
rf_filtered = RandomForestClassifier(random_state=42)
rf_filtered.fit(X_train_filtered, y_train_bc)
filtered_accuracy_bc = rf_filtered.score(X_test_filtered, y_test_bc)

print(f"Accuracy with Filtered Feature Set: {filtered_accuracy_bc:.3f}")

Number of features selected by Filter Method: 15
Accuracy with Full Feature Set: 0.971
Accuracy with Filtered Feature Set: 0.959


### Part 2 — Particle Swarm Optimisation (PSO)

In [16]:
# --- PSO setup for Breast Cancer Dataset ---

# Redefine the objective functions to use the Breast Cancer data splits
# and global variables X_train_bc_scaled, X_test_bc_scaled, y_train_bc, y_test_bc

# Create an instance of the classifier
# Using the same RandomForestClassifier as before for consistency
bc_classifier = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)

def f_per_particle_bc(m, alpha):

    # No need to split again, use pre-defined X_train_bc_scaled, y_train_bc for training
    # and X_test_bc_scaled, y_test_bc for evaluation within the PSO loop's fitness calculation.

    # Get the subset of the features from the binary mask
    if np.count_nonzero(m) == 0:
        X_train_masked = X_train_bc_scaled
        X_test_masked = X_test_bc_scaled
    else:
        X_train_masked = X_train_bc_scaled[:,m==1]
        X_test_masked = X_test_bc_scaled[:,m==1]

    # Perform classification and store performance in P
    bc_classifier.fit(X_train_masked, y_train_bc)
    P = (bc_classifier.predict(X_test_masked) == y_test_bc).mean()

    # Compute for the objective function
    # The denominator for Nf/Nt should be the original number of features for the BC dataset
    j = (alpha * (1.0 - P)
        + (1.0 - alpha) * (1 - (X_train_masked.shape[1] / X_train_bc_scaled.shape[1])))

    return j

def f_bc(M, alpha=0.5):
    """Higher-level method to do classification in the
    whole swarm for Breast Cancer dataset.

    Inputs
    ------
    M: numpy.ndarray of shape (n_particles, dimensions)
        The swarm that will perform the search

    Returns
    -------
    numpy.ndarray of shape (n_particles, )
        The computed loss for each particle
    """

    n_particles = M.shape[0]
    j = [f_per_particle_bc(M[i], alpha) for i in range(n_particles)]
    return np.array(j)


# --- Set-up the optimisation parameters and run the selections algorithm for BC dataset ---

# Define dimensions for the Breast Cancer dataset
bc_dimensions = X_train_bc_scaled.shape[1] # Should be 30

# Initialize a DataFrame to store PSO experiment results for tuning
pso_bc_results = []

# Baseline PSO parameters for Breast Cancer dataset
bc_base_n_particles = 20
bc_base_iters = 15
bc_base_alpha = 0.6 # Start with a slightly higher alpha to encourage feature reduction
bc_base_options = {'c1': 1.5, 'c2': 1.0, 'w': 0.6, 'k': 10, 'p': 2} # Adjusted options for potentially better convergence

print("--- Running PSO Experiments for Breast Cancer Dataset ---")

# Helper function to run PSO experiments for the Breast Cancer dataset
def run_pso_bc_experiment(n_particles, iters, alpha, options_dict, experiment_name):
    print(f"Running experiment: {experiment_name}, Params: n_p={n_particles}, iters={iters}, alpha={alpha}, c1={options_dict['c1']:.2f}, c2={options_dict['c2']:.2f}, w={options_dict['w']:.2f}")
    start_time = time.time()

    current_options = options_dict.copy()
    if n_particles <= current_options['k']:
        current_options['k'] = max(1, n_particles - 1)
        print(f"  Adjusted 'k' to {current_options['k']} for n_particles={n_particles} to prevent IndexError.")

    optimizer = ps.discrete.BinaryPSO(n_particles=n_particles, dimensions=bc_dimensions, options=current_options)
    cost, pos = optimizer.optimize(lambda m: f_bc(m, alpha), iters=iters, verbose=0)
    runtime = time.time() - start_time
    selected_features_count = np.count_nonzero(pos)

    eval_classifier_bc = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)
    subset_performance_bc = 0.0

    if selected_features_count == 0:
        subset_performance_bc = 0.5
        print("  WARNING: PSO selected 0 features. Performance set to 0.5 (random guess).")
    else:
        eval_classifier_bc.fit(X_train_bc_scaled[:,pos==1], y_train_bc)
        subset_performance_bc = (eval_classifier_bc.predict(X_test_bc_scaled[:,pos==1]) == y_test_bc).mean()

    pso_bc_results.append({
        'Experiment': experiment_name,
        'n_particles': n_particles,
        'iters': iters,
        'alpha': alpha,
        'c1': options_dict['c1'],
        'c2': options_dict['c2'],
        'w': options_dict['w'],
        'Runtime (s)': runtime,
        'Selected Features': selected_features_count,
        'Subset Performance': subset_performance_bc,
        'Optimizer Cost': cost
    })
    print(f"  Runtime: {runtime:.2f}s, Sel.Feat: {selected_features_count}, Perf: {subset_performance_bc:.3f}, Cost: {cost:.3f}")
    print("-" * 50)


# --- Run PSO experiments for hyperparameter tuning ---

# Baseline run
run_pso_bc_experiment(bc_base_n_particles, bc_base_iters, bc_base_alpha, bc_base_options, 'BC Baseline')

# Vary n_particles
for n_p in [10, 20, 30]:
    if n_p == bc_base_n_particles: continue
    run_pso_bc_experiment(n_p, bc_base_iters, bc_base_alpha, bc_base_options, f'BC Vary n_particles (n_p={n_p})')

# Vary iters
for i in [10, 15, 20]:
    if i == bc_base_iters: continue
    run_pso_bc_experiment(bc_base_n_particles, i, bc_base_alpha, bc_base_options, f'BC Vary iters (iters={i})')

# Vary alpha
for a in [0.5, 0.7, 0.8]: # Explore alpha values around 0.6
    if a == bc_base_alpha: continue
    run_pso_bc_experiment(bc_base_n_particles, bc_base_iters, a, bc_base_options, f'BC Vary alpha (alpha={a})')

# Vary c1, c2, w slightly (e.g., +/- 0.25 from baseline)
for param in ['c1', 'c2', 'w']:
    original_value = bc_base_options[param]
    for change in [-0.25, 0.25]:
        new_value = round(original_value + change, 2)
        # Skip if new_value is too close to original or already tested
        if abs(new_value - original_value) < 0.01: continue
        temp_options = bc_base_options.copy()
        temp_options[param] = new_value
        run_pso_bc_experiment(bc_base_n_particles, bc_base_iters, bc_base_alpha, temp_options, f'BC Vary {param} ({param}={new_value:.2f})')


print("\n" * 2)
print("--- PSO for Breast Cancer Results Summary ---")
pso_bc_results_df = pd.DataFrame(pso_bc_results)
ordered_cols_bc = ['Experiment', 'n_particles', 'iters', 'alpha', 'c1', 'c2', 'w',
                   'Runtime (s)', 'Selected Features', 'Subset Performance', 'Optimizer Cost']
pso_bc_results_df = pso_bc_results_df[ordered_cols_bc]
print(pso_bc_results_df.round(3).to_markdown(index=False))

# Identify the best performing PSO configuration (e.g., highest performance with fewest features)
# For simplicity, let's look for the highest performance. In a real scenario, you'd balance performance and features.
best_pso_run = pso_bc_results_df.loc[pso_bc_results_df['Subset Performance'].idxmax()]

pso_best_accuracy_bc = best_pso_run['Subset Performance']
pso_best_features_bc = best_pso_run['Selected Features']

print(f"\nBest PSO configuration for Breast Cancer:")
print(f"  Accuracy: {pso_best_accuracy_bc:.3f}")
print(f"  Number of Selected Features: {pso_best_features_bc}")


--- Running PSO Experiments for Breast Cancer Dataset ---
Running experiment: BC Baseline, Params: n_p=20, iters=15, alpha=0.6, c1=1.50, c2=1.00, w=0.60
  Runtime: 10.32s, Sel.Feat: 22, Perf: 0.965, Cost: 0.128
--------------------------------------------------
Running experiment: BC Vary n_particles (n_p=10), Params: n_p=10, iters=15, alpha=0.6, c1=1.50, c2=1.00, w=0.60
  Adjusted 'k' to 9 for n_particles=10 to prevent IndexError.
  Runtime: 6.10s, Sel.Feat: 23, Perf: 0.953, Cost: 0.121
--------------------------------------------------
Running experiment: BC Vary n_particles (n_p=30), Params: n_p=30, iters=15, alpha=0.6, c1=1.50, c2=1.00, w=0.60
  Runtime: 9.45s, Sel.Feat: 24, Perf: 0.965, Cost: 0.101
--------------------------------------------------
Running experiment: BC Vary iters (iters=10), Params: n_p=20, iters=10, alpha=0.6, c1=1.50, c2=1.00, w=0.60
  Runtime: 5.05s, Sel.Feat: 23, Perf: 0.953, Cost: 0.121
--------------------------------------------------
Running experiment: 

In [17]:
# --- PSO setup for Breast Cancer Dataset ---

# Redefine the objective functions to use the Breast Cancer data splits
# and global variables X_train_bc_scaled, X_test_bc_scaled, y_train_bc, y_test_bc

# Create an instance of the classifier
# Using the same RandomForestClassifier as before for consistency
bc_classifier = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)

def f_per_particle_bc(m, alpha):

    # No need to split again, use pre-defined X_train_bc_scaled, y_train_bc for training
    # and X_test_bc_scaled, y_test_bc for evaluation within the PSO loop's fitness calculation.

    # Get the subset of the features from the binary mask
    if np.count_nonzero(m) == 0:
        X_train_masked = X_train_bc_scaled
        X_test_masked = X_test_bc_scaled
    else:
        X_train_masked = X_train_bc_scaled[:,m==1]
        X_test_masked = X_test_bc_scaled[:,m==1]

    # Perform classification and store performance in P
    bc_classifier.fit(X_train_masked, y_train_bc)
    P = (bc_classifier.predict(X_test_masked) == y_test_bc).mean()

    # Compute for the objective function
    # The denominator for Nf/Nt should be the original number of features for the BC dataset
    j = (alpha * (1.0 - P)
        + (1.0 - alpha) * (1 - (X_train_masked.shape[1] / X_train_bc_scaled.shape[1])))

    return j

def f_bc(M, alpha=0.5):
    """Higher-level method to do classification in the
    whole swarm for Breast Cancer dataset.

    Inputs
    ------
    M: numpy.ndarray of shape (n_particles, dimensions)
        The swarm that will perform the search

    Returns
    -------
    numpy.ndarray of shape (n_particles, )
        The computed loss for each particle
    """

    n_particles = M.shape[0]
    j = [f_per_particle_bc(M[i], alpha) for i in range(n_particles)]
    return np.array(j)


# --- Set-up the optimisation parameters and run the selections algorithm for BC dataset ---

# Define dimensions for the Breast Cancer dataset
bc_dimensions = X_train_bc_scaled.shape[1] # Should be 30

# Initialize a DataFrame to store PSO experiment results for tuning
pso_bc_results = []

# Baseline PSO parameters for Breast Cancer dataset
bc_base_n_particles = 20
bc_base_iters = 15
bc_base_alpha = 0.6 # Start with a slightly higher alpha to encourage feature reduction
bc_base_options = {'c1': 1.5, 'c2': 1.0, 'w': 0.6, 'k': 10, 'p': 2} # Adjusted options for potentially better convergence

print("--- Running PSO Experiments for Breast Cancer Dataset ---")

# Helper function to run PSO experiments for the Breast Cancer dataset
def run_pso_bc_experiment(n_particles, iters, alpha, options_dict, experiment_name):
    print(f"Running experiment: {experiment_name}, Params: n_p={n_particles}, iters={iters}, alpha={alpha}, c1={options_dict['c1']:.2f}, c2={options_dict['c2']:.2f}, w={options_dict['w']:.2f}")
    start_time = time.time()

    current_options = options_dict.copy()
    if n_particles <= current_options['k']:
        current_options['k'] = max(1, n_particles - 1)
        print(f"  Adjusted 'k' to {current_options['k']} for n_particles={n_particles} to prevent IndexError.")

    optimizer = ps.discrete.BinaryPSO(n_particles=n_particles, dimensions=bc_dimensions, options=current_options)
    cost, pos = optimizer.optimize(lambda m: f_bc(m, alpha), iters=iters, verbose=0)
    runtime = time.time() - start_time
    selected_features_count = np.count_nonzero(pos)

    eval_classifier_bc = RandomForestClassifier(max_depth=2, n_estimators=10, random_state=42)
    subset_performance_bc = 0.0

    if selected_features_count == 0:
        subset_performance_bc = 0.5
        print("  WARNING: PSO selected 0 features. Performance set to 0.5 (random guess).")
    else:
        eval_classifier_bc.fit(X_train_bc_scaled[:,pos==1], y_train_bc)
        subset_performance_bc = (eval_classifier_bc.predict(X_test_bc_scaled[:,pos==1]) == y_test_bc).mean()

    pso_bc_results.append({
        'Experiment': experiment_name,
        'n_particles': n_particles,
        'iters': iters,
        'alpha': alpha,
        'c1': options_dict['c1'],
        'c2': options_dict['c2'],
        'w': options_dict['w'],
        'Runtime (s)': runtime,
        'Selected Features': selected_features_count,
        'Subset Performance': subset_performance_bc,
        'Optimizer Cost': cost
    })
    print(f"  Runtime: {runtime:.2f}s, Sel.Feat: {selected_features_count}, Perf: {subset_performance_bc:.3f}, Cost: {cost:.3f}")
    print("-" * 50)


# --- Run PSO experiments for hyperparameter tuning ---

# Baseline run
run_pso_bc_experiment(bc_base_n_particles, bc_base_iters, bc_base_alpha, bc_base_options, 'BC Baseline')

# Vary n_particles
for n_p in [10, 20, 30]:
    if n_p == bc_base_n_particles: continue
    run_pso_bc_experiment(n_p, bc_base_iters, bc_base_alpha, bc_base_options, f'BC Vary n_particles (n_p={n_p})')

# Vary iters
for i in [10, 15, 20]:
    if i == bc_base_iters: continue
    run_pso_bc_experiment(bc_base_n_particles, i, bc_base_alpha, bc_base_options, f'BC Vary iters (iters={i})')

# Vary alpha
for a in [0.5, 0.7, 0.8]: # Explore alpha values around 0.6
    if a == bc_base_alpha: continue
    run_pso_bc_experiment(bc_base_n_particles, bc_base_iters, a, bc_base_options, f'BC Vary alpha (alpha={a})')

# Vary c1, c2, w slightly (e.g., +/- 0.25 from baseline)
for param in ['c1', 'c2', 'w']:
    original_value = bc_base_options[param]
    for change in [-0.25, 0.25]:
        new_value = round(original_value + change, 2)
        # Skip if new_value is too close to original or already tested
        if abs(new_value - original_value) < 0.01: continue
        temp_options = bc_base_options.copy()
        temp_options[param] = new_value
        run_pso_bc_experiment(bc_base_n_particles, bc_base_iters, bc_base_alpha, temp_options, f'BC Vary {param} ({new_value:.2f})')


print("\n" * 2)
print("--- PSO for Breast Cancer Results Summary ---")
pso_bc_results_df = pd.DataFrame(pso_bc_results)
ordered_cols_bc = ['Experiment', 'n_particles', 'iters', 'alpha', 'c1', 'c2', 'w',
                   'Runtime (s)', 'Selected Features', 'Subset Performance', 'Optimizer Cost']
pso_bc_results_df = pso_bc_results_df[ordered_cols_bc]
print(pso_bc_results_df.round(3).to_markdown(index=False))

# Identify the best performing PSO configuration (e.g., highest performance with fewest features)
# For simplicity, let's look for the highest performance. In a real scenario, you'd balance performance and features.
best_pso_run = pso_bc_results_df.loc[pso_bc_results_df['Subset Performance'].idxmax()]

pso_best_accuracy_bc = best_pso_run['Subset Performance']
pso_best_features_bc = best_pso_run['Selected Features']

print(f"\nBest PSO configuration for Breast Cancer:")
print(f"  Accuracy: {pso_best_accuracy_bc:.3f}")
print(f"  Number of Selected Features: {pso_best_features_bc}")


--- Running PSO Experiments for Breast Cancer Dataset ---
Running experiment: BC Baseline, Params: n_p=20, iters=15, alpha=0.6, c1=1.50, c2=1.00, w=0.60
  Runtime: 6.59s, Sel.Feat: 25, Perf: 0.971, Cost: 0.084
--------------------------------------------------
Running experiment: BC Vary n_particles (n_p=10), Params: n_p=10, iters=15, alpha=0.6, c1=1.50, c2=1.00, w=0.60
  Adjusted 'k' to 9 for n_particles=10 to prevent IndexError.
  Runtime: 3.85s, Sel.Feat: 24, Perf: 0.947, Cost: 0.112
--------------------------------------------------
Running experiment: BC Vary n_particles (n_p=30), Params: n_p=30, iters=15, alpha=0.6, c1=1.50, c2=1.00, w=0.60
  Runtime: 9.01s, Sel.Feat: 24, Perf: 0.959, Cost: 0.105
--------------------------------------------------
Running experiment: BC Vary iters (iters=10), Params: n_p=20, iters=10, alpha=0.6, c1=1.50, c2=1.00, w=0.60
  Runtime: 5.31s, Sel.Feat: 22, Perf: 0.947, Cost: 0.138
--------------------------------------------------
Running experiment: B

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
print(data.DESCR)

X, y = data.data, data.target

### Comparison Table

Let's fill in the comparison table with the results from the Full Dataset, Filter Method, and the best PSO selection.

| Method             | Accuracy | # Features | Notes                                          |
|:-------------------|:---------|:-----------|:-----------------------------------------------|
| Full Dataset       | 0.971    | 30         | Baseline performance                           |
| Filter Method      | 0.959    | 15         | Selected top 15 features using f_classif       |
| PSO Selection      | 0.971    | 21         | Best performing PSO configuration              |
